# Bayesian A/B Testing — A Hands-On Exercise

*Companion notebook to the trial lecture* **Updating Beliefs with Evidence: Bayes' Theorem from First Principles to Modern Practice**

In this notebook you will:

- Specify a **prior** belief about a website's conversion rate
- Update that belief into a **posterior** after observing real conversion data
- Use **Monte Carlo sampling** to estimate the probability that one design is better than another
- Explore how the choice of prior influences your conclusions

### The scenario

Your team has redesigned the call-to-action button on a landing page. **Design A** is the current version; **Design B** is the proposed redesign. After running both versions in parallel for one week, you have the following data:

| Design | Visitors | Conversions |
|---|---|---|
| A (control) | 1,000 | 100 |
| B (variant) | 1,000 | 120 |

Should the team ship Design B?


## Setup

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Reproducible Monte Carlo sampling
rng = np.random.default_rng(seed=42)

# Observed data
visitors_A, conversions_A = 1000, 100
visitors_B, conversions_B = 1000, 120

print(f"Empirical conversion rate A: {conversions_A / visitors_A:.1%}")
print(f"Empirical conversion rate B: {conversions_B / visitors_B:.1%}")


## Quick recap from the lecture

We model each design's true (unknown) conversion rate $\theta$ with a **Beta prior**:

$$\theta \sim \text{Beta}(\alpha,\, \beta)$$

After observing $k$ conversions in $n$ visits, the posterior is also a Beta distribution — this is the **conjugacy** property we derived in class:

$$\theta \mid \text{data} \;\sim\; \text{Beta}(\alpha + k,\; \beta + n - k)$$

That single update rule is the entire mathematical machinery for today. Everything else is interpretation and computation.


## Step 1 — Specify your prior

Before looking at the data, what do you believe the conversion rate could be?

A `Beta(1, 1)` distribution is **uniform** on $[0, 1]$ — it expresses no preference for any conversion rate. This is a common "weak" or "uninformative" prior, and a good starting point.

**Your task:** define `alpha_prior` and `beta_prior` for a uniform prior, then run the cell to visualize it.


In [ ]:
# TODO: choose values for a uniform Beta(α, β) prior
alpha_prior = ...   # replace ...
beta_prior  = ...   # replace ...

prior = stats.beta(alpha_prior, beta_prior)

# Visualize
theta = np.linspace(0, 1, 500)
plt.figure(figsize=(8, 3))
plt.plot(theta, prior.pdf(theta), linewidth=2,
         label=f"Beta({alpha_prior}, {beta_prior}) prior")
plt.xlabel(r"Conversion rate $\theta$")
plt.ylabel("Density")
plt.title("Prior belief about conversion rate")
plt.legend()
plt.show()


## Step 2 — Compute the posteriors

Apply the conjugate update rule to each design separately.

**Your task:** fill in the four posterior parameters using the formula from the recap.


In [ ]:
# TODO: apply the Beta-Binomial conjugate update for each design
alpha_post_A = ...
beta_post_A  = ...

alpha_post_B = ...
beta_post_B  = ...

posterior_A = stats.beta(alpha_post_A, beta_post_A)
posterior_B = stats.beta(alpha_post_B, beta_post_B)

# Visualize the two posteriors side by side
plt.figure(figsize=(8, 4))
plt.plot(theta, posterior_A.pdf(theta), linewidth=2, label="Posterior A")
plt.plot(theta, posterior_B.pdf(theta), linewidth=2, label="Posterior B")
plt.fill_between(theta, posterior_A.pdf(theta), alpha=0.2)
plt.fill_between(theta, posterior_B.pdf(theta), alpha=0.2)
plt.xlim(0.05, 0.18)
plt.xlabel(r"Conversion rate $\theta$")
plt.ylabel("Density")
plt.title("Posterior beliefs after observing the data")
plt.legend()
plt.show()

# 95% credible intervals
print(f"95% CI for A: [{posterior_A.ppf(0.025):.3f}, {posterior_A.ppf(0.975):.3f}]")
print(f"95% CI for B: [{posterior_B.ppf(0.025):.3f}, {posterior_B.ppf(0.975):.3f}]")


## Step 3 — Estimate $P(\theta_B > \theta_A)$ via Monte Carlo

The posteriors give us full distributions over plausible conversion rates. To answer

> *"How likely is it that B is genuinely better than A?"*

we draw many samples from each posterior and count how often B's sample exceeds A's.

This is a key Bayesian move: instead of a binary "significant / not significant" verdict, we get a **direct probability statement** about the quantity we actually care about.

**Your task:** draw 100,000 samples from each posterior and compute the fraction where B exceeds A.


In [ ]:
n_samples = 100000 # 100

# TODO: draw samples from each posterior
# Hint: posterior_A.rvs(n_samples, random_state=rng)
samples_A = ...
samples_B = ...

# TODO: compute the fraction of samples where B beats A
prob_B_better = ...

print(f"P(B > A | data) = {prob_B_better:.3f}")

# Bonus: distribution of the relative lift
lift = (samples_B - samples_A) / samples_A
plt.figure(figsize=(8, 3))
plt.hist(lift, bins=60, density=True, alpha=0.7)
plt.axvline(0, color='k', linestyle='--', label="no difference")
plt.xlabel("Relative lift  (B − A) / A")
plt.ylabel("Density")
plt.title(f"Posterior distribution of relative lift  (mean = {lift.mean():.1%})")
plt.legend()
plt.show()


## Interpretation

A few things to notice once the cells above run:

- The result is a **probability**, not a p-value. *"There is an X% chance design B has a higher true conversion rate than design A"* is a statement decision-makers can actually use.
- The credible intervals overlap somewhat — yet $P(B > A)$ can be quite high. **Overlap of marginal intervals is not the same as "no difference"**: the joint posterior contains more information than the two marginals shown side by side.
- The expected lift is informative for downstream business decisions (e.g., projected revenue impact).


## Stretch task — Sensitivity to the prior

So far you used `Beta(1, 1)` — deliberately uninformative. In practice you often have *real* prior knowledge: *"historically, this kind of button converts at around 10%, give or take a few points."*

A `Beta(50, 450)` prior has mean $50 / (50 + 450) = 10\%$ and is fairly **confident** — it behaves as if you had already observed 500 visits at a 10% conversion rate before this experiment began.

**Your task:** rerun the analysis with `Beta(50, 450)` as the prior. How does $P(B > A)$ change? Does the conclusion still hold?


In [ ]:
alpha_prior_strong = 50
beta_prior_strong  = 450

# TODO: compute posteriors under the strong prior
alpha_post_A_s = ...
beta_post_A_s  = ...
alpha_post_B_s = ...
beta_post_B_s  = ...

posterior_A_strong = stats.beta(alpha_post_A_s, beta_post_A_s)
posterior_B_strong = stats.beta(alpha_post_B_s, beta_post_B_s)

samples_A_s = posterior_A_strong.rvs(n_samples, random_state=rng)
samples_B_s = posterior_B_strong.rvs(n_samples, random_state=rng)

prob_B_better_strong = np.mean(samples_B_s > samples_A_s)

print(f"With weak prior   Beta(1, 1):    P(B > A) = {prob_B_better:.3f}")
print(f"With strong prior Beta(50, 450): P(B > A) = {prob_B_better_strong:.3f}")

# Visualize side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (pA, pB, label) in zip(
    axes,
    [(posterior_A, posterior_B, "Weak prior"),
     (posterior_A_strong, posterior_B_strong, "Strong prior")]
):
    ax.plot(theta, pA.pdf(theta), label="A", linewidth=2)
    ax.plot(theta, pB.pdf(theta), label="B", linewidth=2)
    ax.set_xlim(0.05, 0.18)
    ax.set_title(label)
    ax.set_xlabel(r"$\theta$")
    ax.legend()
axes[0].set_ylabel("Density")
plt.tight_layout()
plt.show()


## Discussion questions

1. With 1,000 visitors per arm, the data dominates the prior. **At what sample size** would you expect the prior choice to materially change the conclusion?
2. Suppose Design B is more expensive to maintain. How might you incorporate that cost into the decision rule, beyond just checking $P(B > A) > 0.5$?
3. The Beta-Binomial framework assumes each visitor's outcome is **independent and identically distributed**. When might that assumption fail in real-world A/B testing? (Think about repeat visitors, time-of-day effects, or marketing campaigns running in parallel.)


## Solutions

*Try the exercises first — then expand below to check your work.*

<details>
<summary><b>Click to reveal solutions</b></summary>

**Step 1**
```python
alpha_prior = 1
beta_prior  = 1
```

**Step 2**
```python
alpha_post_A = alpha_prior + conversions_A                 # 1 + 100  = 101
beta_post_A  = beta_prior  + visitors_A - conversions_A    # 1 + 900  = 901
alpha_post_B = alpha_prior + conversions_B                 # 1 + 120  = 121
beta_post_B  = beta_prior  + visitors_B - conversions_B    # 1 + 880  = 881
```

**Step 3**
```python
samples_A = posterior_A.rvs(n_samples, random_state=rng)
samples_B = posterior_B.rvs(n_samples, random_state=rng)
prob_B_better = np.mean(samples_B > samples_A)
```
You should obtain $P(B > A) \approx 0.92$.

**Stretch**
```python
alpha_post_A_s = alpha_prior_strong + conversions_A
beta_post_A_s  = beta_prior_strong  + visitors_A - conversions_A
alpha_post_B_s = alpha_prior_strong + conversions_B
beta_post_B_s  = beta_prior_strong  + visitors_B - conversions_B
```
The strong prior pulls both posteriors toward 10%, narrowing the gap and lowering $P(B > A)$ — but with 1,000 visitors per arm, the conclusion that B is probably better still holds. Try it again with only 100 visitors per arm and the prior begins to dominate.

</details>
